## Notebook for testing gridded stat and anomaly methods

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import xarray as xr
import os
from datetime import datetime
import matplotlib.pyplot as plt
import geopandas as gpd



current_path = Path.cwd()

# Search for the utilities directory
for parent in current_path.parents:
    candidate = parent / "utilities"
    if candidate.is_dir():
        sys.path.insert(0, str(parent))
        break
else:
    raise FileNotFoundError("Could not locate 'utilities' directory in parent paths.")

from utilities import init_notebook_environment
#from utilities import statanom_utilities, date_utilities, file_utilities, calc_primprod, calc_phytosizeclass, mapping_utilities,
from utilities import get_dataset_products,get_prod_files,dataset_defaults,run_psc_pipeline,subset_dataset, parse_dataset_info,make_product_output_dir,get_dates,get_source_file_dates
from utilities import period_info
from utilities import get_pyfile_functions,find_function, find_text
from utilities import run_stats_pipeline,get_file_dates,period_regex_debug

env = init_notebook_environment(verbose=False)


In [2]:
from utilities import get_period_sets
files = get_prod_files("CHL")
dates = get_file_dates(files)
start_dates = [start for start, _ in dates]

sets = get_period_sets('SEA',dates=start_dates)

for a in sets:
    print("SEA periods:",a)

📦 Found 454 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/SOURCE/GLOBAL_4KM_DAILY/CHL
SEA periods: SEA_19980101_19980331
SEA periods: SEA_19980401_19980630
SEA periods: SEA_19980701_19980930
SEA periods: SEA_19981001_19981231
SEA periods: SEA_19990101_19990331
SEA periods: SEA_19990401_19990630
SEA periods: SEA_19990701_19990930
SEA periods: SEA_19991001_19991231
SEA periods: SEA_20000101_20000331
SEA periods: SEA_20000401_20000630
SEA periods: SEA_20000701_20000930
SEA periods: SEA_20001001_20001231
SEA periods: SEA_20010101_20010331
SEA periods: SEA_20010401_20010630
SEA periods: SEA_20010701_20010930
SEA periods: SEA_20011001_20011231
SEA periods: SEA_20020101_20020331
SEA periods: SEA_20020401_20020630
SEA periods: SEA_20020701_20020930
SEA periods: SEA_20021001_20021231
SEA periods: SEA_20030101_20030331
SEA periods: SEA_20030401_20030630
SEA periods: SEA_20030701_20030930
SEA periods: SEA_20031001_20031231
SEA periods: SEA_20040101_20040331
SEA periods: 

In [7]:
from utilities import get_period_info
info = get_period_info()
periods = info.keys()
for p in periods:
    i = info.get(p)
    print(f"{p} is climatology = {i['is_climatology']}")

D is climatology = False
DD is climatology = False
DOY is climatology = True
D3 is climatology = False
DD3 is climatology = False
D8 is climatology = False
DD8 is climatology = False
W is climatology = False
WW is climatology = False
WEEK is climatology = True
M is climatology = False
MM is climatology = False
M3 is climatology = False
MM3 is climatology = False
MONTH is climatology = True
SEA is climatology = False
SEASON is climatology = True
A is climatology = False
AA is climatology = False
ANNUAL is climatology = True
Y is climatology = False
YY is climatology = False
YEAR is climatology = True


In [3]:
from utilities import get_period_dates

files = ["D_20250926",
         "DD_20250926_20251231",
         "DOY_2020_2025",
         "DD3_20250310_20250312",
         "D8_20250101_20250108",
         "W_202301",
         "WW_202501_202552",
         "WEEK_2020_2025",
         "M_202204",
         "MM_202501_202612",
         "MM3_202501_202512",
         "MONTH_2020_2025",
         "A_2025",
         "ANNUAL_2020_2025",
         "Y_2025",
         "YY_2020_2025",
         "YEAR_2020_2025"
         ]

#g = get_period_dates(files)
for f in files:
    print(f"{f}: {get_file_dates([f])}")

dates = get_file_dates(files)

date_format="%Y%m%d"
valid = [
        (datetime.strptime(s, date_format), datetime.strptime(e, date_format))
        for s, e in dates
        if s != "NA" and e != "NA"
    ]

min_date = min(s for s, _ in valid)
max_date = max(e for _, e in valid)
print (min_date.strftime(date_format), max_date.strftime(date_format))




D_20250926: [('20250926', '20250926')]
DD_20250926_20251231: [('20250926', '20251231')]
DOY_2020_2025: [('20200101', '20251231')]
DD3_20250310_20250312: [('20250310', '20250312')]
D8_20250101_20250108: [('20250101', '20250101')]
W_202301: [('20230102', '20230108')]
WW_202501_202552: [('20241230', '20251228')]
WEEK_2020_2025: [('20200101', '20251231')]
M_202204: [('20220401', '20220430')]
MM_202501_202612: [('20250101', '20261231')]
MM3_202501_202512: [('20250101', '20251231')]
MONTH_2020_2025: [('20200101', '20251231')]
A_2025: [('20250101', '20251231')]
ANNUAL_2020_2025: [('20200101', '20251231')]
Y_2025: [('20250101', '20251231')]
YY_2020_2025: [('20200101', '20251231')]
YEAR_2020_2025: [('20200101', '20251231')]
20200101 20261231


In [4]:
filename = "WW_202501_202552"
matches = period_regex_debug(filename)

for code, matched, m in matches:
    if matched:
        print(f"{code}: ✅ matched → {m.groupdict()}")
    else:
        print(f"{code}: ❌ no match")

D: ❌ no match
DD: ❌ no match
DOY: ❌ no match
D3: ❌ no match
DD3: ❌ no match
D8: ❌ no match
DD8: ❌ no match
W: ❌ no match
WW: ✅ matched → {'start': '202501', 'end': '202552'}
WEEK: ❌ no match
M: ❌ no match
MM: ❌ no match
M3: ❌ no match
MM3: ❌ no match
MONTH: ❌ no match
SEA: ❌ no match
SEASON: ❌ no match
A: ❌ no match
AA: ❌ no match
ANNUAL: ❌ no match
Y: ❌ no match
YY: ❌ no match
YEAR: ❌ no match


In [2]:
run_stats_pipeline(prods='SST',dataset='CORALSST',periods=['M','MONTH'])

NameError: name 'run_stats_pipeline' is not defined

### Daily inputs
Create stats from the daily input files
* Monthly
* Weekly
* 3-day running
* 8-day running

In [ ]:
# Get Chlorophyll Files
get_dataset_products('OCCCI')
print(dataset_defaults()['OCCCI'])
dataset_products = get_dataset_products('OCCCI')
print(dataset_products.keys())
files = get_prod_files('CHL',dataset='OCCCI')
ds = xr.open_mfdataset(files)
ds.data_vars.keys()

('V6.0', 'GLOBAL_4KM', 'CHL')
dict_keys(['EDAB_PRODUCTS', 'SOURCE'])
📦 Found 454 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/SOURCE/GLOBAL_4KM_DAILY/CHL


KeysView(<xarray.Dataset> Size: 678GB
Dimensions:             (time: 454, lat: 4320, lon: 8640)
Coordinates:
  * lat                 (lat) float64 35kB 89.98 89.94 89.9 ... -89.94 -89.98
  * lon                 (lon) float64 69kB -180.0 -179.9 -179.9 ... 179.9 180.0
  * time                (time) datetime64[ns] 4kB 1998-01-01 ... 2005-01-09
Data variables:
    MERIS_nobs          (time, lat, lon) float32 68GB dask.array<chunksize=(1, 270, 270), meta=np.ndarray>
    MODISA_nobs         (time, lat, lon) float32 68GB dask.array<chunksize=(1, 270, 270), meta=np.ndarray>
    OLCI-A_nobs         (time, lat, lon) float32 68GB dask.array<chunksize=(1, 270, 270), meta=np.ndarray>
    OLCI-B_nobs         (time, lat, lon) float32 68GB dask.array<chunksize=(1, 270, 270), meta=np.ndarray>
    SeaWiFS_nobs        (time, lat, lon) float32 68GB dask.array<chunksize=(1, 270, 270), meta=np.ndarray>
    VIIRS_nobs          (time, lat, lon) float32 68GB dask.array<chunksize=(1, 270, 270), meta=np.ndarray>

In [ ]:
from utilities import get_file_dates,get_period_dates

#psc = get_prod_files('PSC',period='M')
#chl = get_prod_files('CHL')

#files = [psc[0],chl[0],psc[1],chl[1]]
#dates = get_file_dates(files)







[('NA', 'NA'), ('20220310', '20220312'), ('NA', 'NA'), ('NA', 'NA')]

In [10]:
files = [
    "M_202204-other.nc",
    "/data/OCCCI/CHL_202205.nc",
    "DD3_20220310_20220312-EDAB.nc",
    "OISST.198109.mean.1981.nc"
]

date_ranges = get_file_dates(
    files,
    source_format="yyyy-mm-dd",
    period_format="%Y%m%d",
    placeholder="NA"
)

date_ranges

[('NA', 'NA'), ('NA', 'NA'), ('NA', 'NA'), ('1981-01-01', '1981-01-01')]